In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import rasterio
import geopandas as gpd
from shapely.geometry import Point
import contextily as cxt

import os
from glob import glob
from tqdm.notebook import tqdm

# Utiliy for the efficiency improvement
import cupy as cp
from rasterio.windows import Window
from collections import defaultdict
from datetime import datetime
import sys, traceback
from rasterio.transform import xy
from rasterio.windows import transform
from pyproj import CRS

In [ ]:
data_dir = r"D:/ForestFire/CBH/data"
raster_dir = r"F:\CBH"

nifs_cbh = os.path.join(raster_dir, r"F:\CBH\NIFS_CBH_M.tif")
# WP-B (2026-06-24): repointed to the validated plot-disjoint map. The old CBH4.tif was INVALID
# (slope-feature + CBH-formula bugs) and was DELETED — see .claude/rules/changelog.md.
# NOTE: this notebook flow is SUPERSEDED by src/wp_b5_category_table.py + src/wp_b5_boxplots.py;
# downstream filenames now carry the _plotdisjoint tag (was the stale _CR4 tag); the canonical
# plot-disjoint results come from src/wp_b5_category_table.py + src/wp_b5_boxplots.py.
pred_cbh = os.path.join(raster_dir, "CBH_meter_hybrid_model.tif")

In [ ]:
# ===== read table data & convert it into the Point =====
df_nfi7 = pd.read_csv(os.path.join(data_dir, r"NFI7_cleaned2.csv"))
df_nfi7['geometry'] = df_nfi7.apply(lambda row: Point(row['Long'], row['Lat']), axis=1) # create geometry
gdf_nfi7 = gpd.GeoDataFrame(df_nfi7, geometry='geometry', crs='EPSG:5174') # save it into geodataframe

In [ ]:
def write_log(f_prefix, log_content, initialize_log=False):
    """
    로그 파일 생성 및 작성 함수
    """
    curr_dir = os.getcwd()
    if initialize_log:
        mode = 'w'
    else:
        mode = 'a'
    with open(os.path.join(curr_dir, f'log_{f_prefix}.txt'), mode, encoding='utf-8') as f:
        f.write(str(log_content) + '\n')

# Create the error map

In [ ]:
# ===== Get metatdata from rasters =====
default_block_size = 512
STOP_ON_ERROR = True

with rasterio.open(nifs_cbh) as nifs, \
 rasterio.open(pred_cbh) as pred:
    ref_height, ref_width = nifs.height, nifs.width
    ref_transform = nifs.transform
    profile = nifs.profile.copy()
    b_height, b_width = profile.get('blockysize', default_block_size), profile.get('blockxsize', default_block_size)
    block_cnt = int(np.ceil(ref_height / b_height)) * int(np.ceil(ref_width / b_width))
    nifs_nodata = nifs.nodata if nifs.nodata else -9999.
    pred_nodata = pred.nodata if pred.nodata else -9999.

    # Check: all rasters must align
    ja = CRS.from_wkt(nifs.crs.to_wkt()).to_json_dict()
    jb = CRS.from_wkt(pred.crs.to_wkt()).to_json_dict()
    assert ja['id']['code'] == jb['id']['code']
    assert nifs.transform == pred.transform
    assert (nifs.width, nifs.height) == (pred.width, pred.height)

    # update profile to process the large data
    profile.update(driver = 'GTiff', dtype = 'float32', count=1, nodata=pred_nodata,
                  tiled=True, blockxsize=b_width, blockysize=b_height,
                  compress = 'ZSTD', predictor=3, bigtiff='YES')

    dst_files = {
        suffix : rasterio.open(os.path.join(raster_dir, f"{suffix}.tif"), 'w', **profile)
        for suffix in ['Deviation']
    }
    print("All the files are ready!")


    # ===== Process rasters by blocks =====
    try:
        # iterate by windows; remove [:100 ] & list() to do full
        for ji, win in tqdm(list(pred.block_windows(1)), desc="Creating Raster...", total = block_cnt):
            try:
                # ===== read blocks to GPU =====
                nifs_block = cp.asarray(nifs.read(1, window=win))
                pred_block = cp.asarray(pred.read(1, window=win))
                nifs_flat = nifs_block.ravel()
                pred_flat = pred_block.ravel()
                # ===== allocate outputs =====
                out_dtype = cp.float32
                out_ras = cp.full(pred_block.shape, pred_nodata, dtype=out_dtype)
                
                mask = (nifs_flat != nifs_nodata) & (pred_flat != pred_nodata)
                out_ras.ravel()[mask] = cp.asarray(pred_flat[mask] - nifs_flat[mask], dtype=out_dtype)

                dst_files['Deviation'].write(out_ras.get(), 1, window=win)
                
            # =====Exception: write nodata and continue =====
            except Exception as e:
                write_log('evaluation', f"BLOCK {ji} failed: {type(e).__name__}: {e}")
                traceback.print_exc()
                if STOP_ON_ERROR: raise
                else:
                    try:
                        dst_files['Deviation'].write(out_ras.get(), 1, window=win)
                    except Exception:
                        pass
                    continue
    # ===== Finally: Always Run this code despite excpetions =====
    finally:
        for f in dst_files.values():
            f.close()

# 수종별 평가

In [ ]:
from collections import defaultdict
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.spatial.distance import jensenshannon

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from collections import defaultdict
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.spatial.distance import jensenshannon

def analyze_rasters_specieswise(
    pred_path=None, ref_path=None, species_path=None,
    mode="single",
    bins=512, nodata=None, vmin=0, vmax=30
):
    """
    pred_path: 지하고 예측값(y_hat)이 저장된 레스터
    ref_path: 지하고 관측값(y)이 저장된 레스터
    species path: 수종 정보가 저장된 레스터
    mode:
      - "single": 단일 raster → mean, median, IQR, std
      - "pairwise": raster vs raster → RMSE, MAE, Bias, R2, CCC
    bins: 히스토그램 추정 시 bin의 개수
    nodata: 레스터 마스킹 시 사용하는 nodata 값
    vmin: 히스토그램 추정 시 최댓값
    vmax: 히스토그램 추정 시 최솟값
    """

    with rasterio.open(pred_path) as rpred, \
         rasterio.open(species_path) as rsid:
        default_block_size = 512
        profile = rpred.profile
        b_height = profile.get('blockysize', default_block_size)
        b_width = profile.get('blockxsize', default_block_size)
        height, width =  rpred.height, rpred.width
        block_cnt = int(np.ceil(height / b_height)) * int(np.ceil(width / b_width))
        pred_nodata, sid_nodata = rpred.nodata, rsid.nodata
             
        if mode == "pairwise":
            if ref_path is None:
                raise ValueError("pairwise mode requires ref_path")
            rref = rasterio.open(ref_path)
            ref_nodata = rref.nodata
            assert rpred.shape == rref.shape == rsid.shape

        edges = np.linspace(vmin, vmax, bins+1) # 히스토그램 추정 시 사용할 구간 값들

        # n: 데이터 개수, sum_(var): 합계, sum_(var)2: 제곱합, hist_(var): 히스토그램 추정
        # sum_e: 편차의 합, sum_abs_e: 편차 절댓값의 합, sum_sq_e: 편차 제곱의 합
        acc = defaultdict(lambda: {
            "n":0,
            "sum_pred":0.0, "sum_pred2":0.0,
            "hist_pred": np.zeros(bins, dtype=np.int64),
            "sum_ref":0.0, "sum_ref2":0.0, "sum_predref":0.0,
            "sum_e":0.0, "sum_abs_e":0.0, "sum_sq_e":0.0,
        })

        for _, window in tqdm(rpred.block_windows(1), desc="Processing...", total = block_cnt):
            pred = rpred.read(1, window=window, masked=True).astype(float) # masked=True: nodata 자동으로 마스킹 처리...왜 이제 알려줘?
            sid = rsid.read(1, window=window, masked=True)

            # pred raster & species raster로 마스크 생성
            valid = ~pred.mask & np.isfinite(pred)
            if pred_nodata is not None and np.isfinite(pred_nodata):
                valid &= (pred != pred_nodata)
            valid &= ~sid.mask
            if sid_nodata is not None:
                valid &= (sid != sid_nodata)

            # pairwise일 경우, true raster로도 mask 생성
            if mode == "pairwise":
                y_true = rref.read(1, window=window, masked=True).astype(float)
                valid = ~y_true.mask & np.isfinite(y_true)
                if ref_nodata is not None and np.isfinite(ref_nodata):
                    valid &= (y_true != ref_nodata)
                    
            if not valid.any():
                continue

            species_ids = np.unique(sid[valid].astype(int))
            for sp in species_ids:
                mask = valid & (sid.astype(int) == sp)
                if not mask.any():
                    continue
                pred_masked = pred[mask]
                acc_sp = acc[int(sp)]
                acc_sp["n"] += pred_masked.size
                acc_sp["sum_pred"] += pred_masked.sum()
                acc_sp["sum_pred2"] += (pred_masked**2).sum()

                # masked된 array의 값들이 어느 구간(edge)에 속하는지 반환하는 line (-1을 하면 bin의 번호)
                # idx: pred가 속하는 BIN의 index번호
                idx = np.searchsorted(edges, np.clip(pred_masked, vmin, vmax), side="right")-1
                # np.clip()에 나오는 index를 기준으로 acc_sp에 해당 index에 "1"씩 더하는 함수(중복합산에 유리함)
                # 즉, 각 bin에 속하는 값들의 count를 구하는 함수
                np.add.at(acc_sp["hist_pred"], np.clip(idx,0,bins-1), 1) #일반적인 표현: hist[idx] += 1

                if mode == "pairwise":
                    y_true_masked = y_true[mask]
                    acc_sp["sum_ref"] += y_true_masked.sum()
                    acc_sp["sum_ref2"] += (y_true_masked**2).sum()
                    acc_sp["sum_predref"] += (pred_masked*y_true_masked).sum()
                    e = pred_masked - y_true_masked
                    acc_sp["sum_e"] += e.sum()
                    acc_sp["sum_abs_e"] += np.abs(e).sum()
                    acc_sp["sum_sq_e"] += (e**2).sum()

        # ----- 최종 계산 -----
        print("Calculate the results...")
        results = {}
        for sp, a in acc.items():
            if a["n"] == 0:
                continue

            mean_pred = a["sum_pred"]/a["n"]
            var_pred = a["sum_pred2"]/a["n"] - mean_pred**2
            std_pred = np.sqrt(max(var_pred,0))

            if mode=="single":
                c = a["hist_pred"].cumsum()
                # histogram을 활용하여 분위수를 계산하는 함수
                def q(p):
                    k = np.searchsorted(c, p*c[-1]) # p*c[-1]: p 위치의 순위
                    return 0.5*(edges[k]+edges[min(k+1,bins)]) # k번째 값이 속하는 구간의 중위값
                med = q(0.5); iqr = q(0.75)-q(0.25)
                results[sp] = dict(mean=mean_pred, std=std_pred,
                                   median=med, IQR=iqr)

            elif mode=="pairwise":
                mean_ref = a["sum_ref"]/a["n"]
                var_ref = a["sum_ref2"]/a["n"] - mean_ref**2
                std_ref = np.sqrt(max(var_ref,0))

                rmse = np.sqrt(a["sum_sq_e"]/a["n"])
                mae  = a["sum_abs_e"]/a["n"]
                bias = a["sum_e"]/a["n"]
                sst  = a["sum_ref2"] - a["n"]*mean_ref**2
                r2   = 1 - a["sum_sq_e"]/sst if sst>0 else np.nan # (1 - SSE / SST)
                rho  = (a["sum_predref"]/a["n"] - mean_pred*mean_ref)/(std_pred*std_ref)
                denom = var_pred+var_ref+(mean_pred-mean_ref)**2
                ccc  = (2*rho*std_pred*std_ref/denom) if denom>0 else np.nan

                results[sp] = dict(RMSE=rmse, MAE=mae, Bias=bias, R2=r2, CCC=ccc) # 이중 dictionary

        return results, acc, edges


In [ ]:
def json_dump_defaultdict(path, obj):
    def convert(o):
        # 1) defaultdict → dict
        if isinstance(o, defaultdict):
            return dict(o)
        # 2) NumPy → native / list
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        # 3) sets / tuples etc.
        if isinstance(o, (set, tuple)):
            return list(o)
        # Anything else: let json raise
        raise TypeError(f"{type(o).__name__} not JSON serializable")

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, default=convert, ensure_ascii=False, indent=2)

In [ ]:
species_path = os.path.join(raster_dir, "KOFTR_INT.tif")
if os.path.exists(species_path):
    print("File exists")

In [ ]:
# single: predicted cbh
stats_pred, acc_pred, edges_pred = analyze_rasters_specieswise(
    pred_path=pred_cbh, ref_path=None, species_path=species_path,
    mode="single",
    bins=512, nodata=None, vmin=0, vmax=30
)

In [ ]:
# save into JSON
import json
save_dir = r"D:/ForestFire/CBH/result/Baseline3"
json_dump_defaultdict(os.path.join(save_dir, "stats_prediction_plotdisjoint.txt"), acc_pred)

In [ ]:
# save into dataframe
df = pd.DataFrame.from_dict(stats_pred, orient="index")
df.to_csv(os.path.join(save_dir, "stats_prediction_plotdisjoint.csv"))

In [ ]:
# single: NIFS cbh
stats_nifs, acc_nifs, edges_nifs = analyze_rasters_specieswise(
    pred_path=nifs_cbh, ref_path=None, species_path=species_path,
    mode="single",
    bins=512, nodata=None, vmin=0, vmax=30
)

In [ ]:
# save into dataframe
save_dir = r"D:/ForestFire/CBH/result/Baseline3"
json_dump_defaultdict(os.path.join(save_dir, "stats_NIFS_plotdisjoint.txt"), acc_nifs)

In [ ]:
df = pd.DataFrame.from_dict(stats_nifs, orient="index")
df.to_csv(os.path.join(save_dir, "NIFS_stats_plotdisjoint.csv"))

In [ ]:
# pairwise: NIFS & Prediction
compare_pred_nifs, _, _ = analyze_rasters_specieswise(
    pred_path=pred_cbh, ref_path=nifs_cbh, species_path=species_path,
    mode="pairwise",
    bins=512, nodata=None, vmin=0, vmax=30
)

In [ ]:
df = pd.DataFrame.from_dict(compare_pred_nifs, orient="index")
df.to_csv(os.path.join(save_dir, "NIFS-pred_compare_plotdisjoint.csv"))

In [ ]:
size = len(acc_pred)
w = int(np.ceil(np.sqrt(size)))
h = int(np.ceil(size / w))

fig, axes = plt.subplots(h, w, figsize=(12, 10))
axes = axes.flatten()

for ax, (key, info) in zip(axes, acc_pred.items()):
    values1 = info['hist_pred']
    values2 = acc_nifs[key]['hist_pred']
    ax.hist(values1, bins=20, color="blue", alpha=.8, label="Prediction")
    ax.hist(values2, bins=20, color="red", alpha=.5, label="NIFS")
    ax.set_title(f"Histogram({key})")
for ax in axes[len(acc_pred):]:
    ax.axis('off')
    
fig.legend(labels=['Prediction', 'NIFS'],
          loc="upper right",
          ncol=2,
          frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# 저장하기
for (key, info) in tqdm(acc_pred.items()):
    plt.figure(figsize=(8, 5))
    plt.hist(values1, bins=10, color="skyblue", alpha=.8, label='prediction')
    plt.hist(values2, bins=10, color="pink", alpha=.5, label='NIFS')
    plt.legend(labels=['prediction', 'NIFS'],
              loc='upper right',
              ncol=2,
              frameon=True)
    plt.savefig(os.path.join(r"D:/ForestFire/CBH/fig", f"histogram_{key}_pred+nifs.png"), dpi=300, bbox_inches='tight')
    plt.close()
    

In [ ]:
all_species_pred = np.full_like(acc_pred[10]['hist_pred'], fill_value=0.)
all_species_nifs = np.full_like(acc_pred[10]['hist_pred'], fill_value=0.)
for (key, info) in tqdm(acc_pred.items()):
    all_species_pred += info['hist_pred']
    all_species_nifs += acc_nifs[key]['hist_pred']

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(all_species_pred, bins=50, color="skyblue", alpha=.8, label='prediction')
plt.hist(all_species_nifs, bins=50, color="pink", alpha=.5, label='NIFS')
plt.legend(labels=['prediction', 'NIFS'],
          loc='upper right',
          ncol=2,
          frameon=True)

In [ ]:
def distribution_raster_table(pred_path, species_path, ref_table,
                                 species_col="species", obs_col="obs",
                                 bins=512, nodata=None, vmin=0, vmax=30):
    """
    Compare raster-predicted distributions vs NFI table distributions (species-wise).
    
    Parameters
    ----------
    pred_path : str
        Path to prediction raster
    species_path : str
        Path to species raster (same shape)
    ref_table : DataFrame
        NFI table with columns [species_col, obs_col]
    """
    # 1_historgram 구간 설정
    edges = np.linspace(vmin, vmax, bins+1)

    with rasterio.open(pred_path) as rpred, \
         rasterio.open(species_path) as rsid:
            pred_nodata, sid_nodata = rpred.nodata, rsid.nodata
            assert rpred.shape == rsid.shape
            # 2_histogram 구간별 default dictionary 만드는 함수
            # key가 없는 dictionary을 만드는 함수
            # 새로운 key가 들어오면, lambda를 호출하여 {새로운 key : [0, 0, 0, 0, ...]}을 만들게 됨 (keyerror 발생 X)
            hist_pred = defaultdict(lambda: np.zeros(bins, dtype=np.int64)) 
            height, width = rpred.height, rpred.width
            b_h, b_w = rpred.profile.get('blockysize', 512), rpred.profile.get('blockxsize', 512)
            block_cnt = int(np.ceil(height / b_h)) *  int(np.ceil(width / b_w))
            for _, window in tqdm(rpred.block_windows(1), desc="Block-wise rater reading...", leave=False, total=block_cnt):
                pred = rpred.read(1, window=window, masked=True).astype(float)
                sid = rsid.read(1, window=window, masked=True).astype(int)
    
                # 유효 마스크 생성
                valid = ~pred.mask & np.isfinite(pred)
                if pred_nodata is not None and np.isfinite(pred_nodata):
                    valid &= (pred != pred_nodata)
                valid &= ~sid.mask & np.isfinite(sid)
                if sid_nodata is not None and np.isfinite(sid_nodata):
                    valid &= (sid != sid_nodata)
                if not valid.any():
                    continue
    
                for sp in np.unique(sid[valid]):
                    mask = valid & (sid == sp)
                    vals = pred[mask]
                    idx = np.searchsorted(edges, np.clip(vals, vmin, vmax), side="right") - 1
                    np.add.at(hist_pred[int(sp)], np.clip(idx, 0, bins-1), 1)
    
            # 2) species별 NFI obs 준비 ---
            results = []
            for sp, g in tqdm(ref_table.groupby(species_col), desc="Calculating...", leave=True):
                obs = g[obs_col].dropna().to_numpy(dtype=float) # 데이터 nan값 & 형 변환 확실하게!
                # NFI나 Raster의 값이 없으면,
                if obs.size == 0 or hist_pred[int(sp)].sum() == 0:
                    results.append({"species": sp, "KS_D": np.nan, "KS_p": np.nan,
                                    "Wasserstein": np.nan, "JS": np.nan})
                    continue
        
                # 히스토그램 기반 근사로 변환: raster 분포 (확률화)
                # KS, JS, Wasserstein 모두 확률질량(합=1) 분포를 입력으로 써야 정확함
                
                # mids: bins의 중앙값 → 값이 속하는 위치 추정
                # wp: 분포의 확률 가중치(각 bin에 속할 확률) → 값이 나올 확률(질량, weight)
                hp = hist_pred[int(sp)]
                mids = 0.5 * (edges[:-1] + edges[1:]) # edges[:-1]: 왼쪽 경계, edges[1:]: 오른쪽 경계 (참고: edges[::-1]: 역순정렬)
                wp = hp / hp.sum()
        
                # obs 히스토그램
                hist_range = (min(mids.min(), obs.min()), max(mids.max(), obs.max()))
                # ho: 각 bin의 확률밀도(density=True / False: 데이터 개수), _: 경계값 배열
                ho, edges_obs = np.histogram(obs, bins=bins, range=hist_range, density=True)
                bin_widths = np.diff(edges)
                # 확률질량으로 변환해서 계산
                po = ho * bin_widths
                po = po / po.sum()
        
                # KS (샘플 필요 → obs vs raster 샘플 근사)
                # 간단하게는 obs vs raster histogram 누적비교
                cp = np.cumsum(wp)
                co = np.cumsum(po)
                ks_d = float(np.max(np.abs(cp - co)))
                ks_p = np.nan  # 히스토그램 기반이므로 정확 p-value는 어려움
        
                # Wasserstein (obs 샘플 vs raster 분포 근사)
                wd = float(wasserstein_distance(mids, mids, u_weights=wp, v_weights=po))
        
                # JS-divergence
                eps = 1e-12
                p1 = (wp + eps); p1 = p1 / p1.sum()
                p2 = (po + eps); p2 = p2 / p2.sum()
                js = float(jensenshannon(p1, p2))
                
                results.append({
                    "species": sp,
                    "KS_D": ks_d, "KS_p": ks_p,
                    "Wasserstein": wd, "JS": js,
                    "mean_pred": np.sum(mids * wp),
                    "mean_obs": obs.mean(),
                    "median_obs": np.median(obs), "n_obs": obs.size
                })

    return pd.DataFrame(results)

In [ ]:
ft_to_m = 0.3048
df_nfi7['CBH(m)'] = df_nfi7['CBH(ft)'] * ft_to_m

In [ ]:
# pairwise: Prediction & NFI
compare_pred_nfi =  distribution_raster_table(pred_path=pred_cbh, species_path=species_path, ref_table=df_nfi7,
                                 species_col="SID", obs_col="CBH(m)",
                                 bins=512, nodata=None, vmin=0, vmax=30)

In [ ]:
compare_pred_nfi.to_csv(os.path.join(save_dir, "NFI-pred_compare_plotdisjoint.csv"))

In [ ]:
# pairwise: NIFS & NFI
compare_nifs_nfi =  distribution_raster_table(pred_path=nifs_cbh, species_path=species_path, ref_table=df_nfi7,
                                 species_col="SID", obs_col="CBH(m)",
                                 bins=512, nodata=None, vmin=0, vmax=30)

In [ ]:
compare_nifs_nfi.to_csv(os.path.join(save_dir, "NFI-NIFS_compare_plotdisjoint.csv"))

In [ ]:
# Prediction 결과를 NIFS 결과 기준으로 분류해서 대표값 산출
def analyze_raster_categorywise(
    pred_path=None, species_path=None, age_path=None, dbh_path=None,
    bins=512, nodata=None, vmin=0, vmax=30
):
    """
    pred_path: 지하고 예측값(y_hat)이 저장된 레스터
    age_path: 
    dbh_path:
    species path: 수종 정보가 저장된 레스터
    bins: 히스토그램 추정 시 bin의 개수
    nodata: 레스터 마스킹 시 사용하는 nodata 값
    vmin: 히스토그램 추정 시 최댓값
    vmax: 히스토그램 추정 시 최솟값
    """
    # need to add the combination of species-age-dbh
    species_list =  [11, 12, 14, 15, 10, 31, 33, 32, 30, 77]
    age_dbh_list = [[2, 1], [3, 1], [3, 2], [4, 1], [4, 2], [4, 3], [5, 1],
                   [5, 2], [5, 3], [6, 1], [6, 2], [6, 3], [7, 1], [7, 2],
                   [7, 3],  [8, 2], [8, 3], [9, 2], [9, 3]]
    spe_age_dbh = [[i] + j for i in species_list for j in age_dbh_list]
    
    with rasterio.open(pred_path) as rpred, \
         rasterio.open(species_path) as rsid, \
         rasterio.open(age_path) as rage, \
         rasterio.open(dbh_path) as rdbh:
             
        default_block_size = 512
        profile = rpred.profile
        b_height = profile.get('blockysize', default_block_size)
        b_width = profile.get('blockxsize', default_block_size)
        height, width =  rpred.height, rpred.width
        block_cnt = int(np.ceil(height / b_height)) * int(np.ceil(width / b_width))
        pred_nodata, sid_nodata, age_nodata, dbh_nodata = rpred.nodata, rsid.nodata, rage.nodata, rdbh.nodata

        assert rpred.shape == rsid.shape == rage.shape == rdbh.shape

        edges = np.linspace(vmin, vmax, bins+1) # 히스토그램 추정 시 사용할 구간 값들

        # n: 데이터 개수, sum_(var): 합계, sum_(var)2: 제곱합, hist_(var): 히스토그램 추정
        # sum_e: 편차의 합, sum_abs_e: 편차 절댓값의 합, sum_sq_e: 편차 제곱의 합
        acc = defaultdict(lambda: {
            "n":0,
            "sum_pred":0.0,
            "min" : 0.0,
            "max" : 0.0,
            "hist_pred": np.zeros(bins, dtype=np.int64)
        })

        for _, window in tqdm(rpred.block_windows(1), desc="Processing...", leave=False, total = block_cnt):
            pred = rpred.read(1, window=window, masked=True).astype(float) # masked=True: nodata 자동으로 마스킹 처리...왜 이제 알려줘?
            sid = rsid.read(1, window=window, masked=True)
            age = rage.read(1, window=window, masked=True)
            dbh = rdbh.read(1, window=window, masked=True)

            # pred raster & species raster로 마스크 생성
            valid = ~pred.mask & np.isfinite(pred)
            if pred_nodata is not None and np.isfinite(pred_nodata):
                valid &= (pred != pred_nodata)
            valid &= ~sid.mask
            if sid_nodata is not None:
                valid &= (sid != sid_nodata)
            valid &= ~age.mask
            if age_nodata is not None:
                valid &= (age != age_nodata)
                    
            if not valid.any():
                continue


            for cond in spe_age_dbh:
                mask = valid & (sid.astype(int) == cond[0]) & (age.astype(int) == cond[1]) & (dbh.astype(int) == cond[2])
                if not mask.any():
                    continue
                pred_masked = pred[mask]
                dict_key = f"{cond[0]}-{cond[1]}-{cond[2]}"
                acc_sp = acc[dict_key]
                acc_sp["n"] += pred_masked.size
                acc_sp["sum_pred"] += pred_masked.sum()
                acc_sp["min"] = np.minimum(acc_sp["min"], np.min(pred_masked))
                acc_sp["max"] = np.maximum(acc_sp["max"], np.max(pred_masked))

                # masked된 array의 값들이 어느 구간(edge)에 속하는지 반환하는 line (-1을 하면 bin의 번호)
                # idx: pred가 속하는 BIN의 index번호
                idx = np.searchsorted(edges, np.clip(pred_masked, vmin, vmax), side="right")-1
                # np.clip()에 나오는 index를 기준으로 acc_sp에 해당 index에 "1"씩 더하는 함수(중복합산에 유리함)
                # 즉, 각 bin에 속하는 값들의 count를 구하는 함수
                np.add.at(acc_sp["hist_pred"], np.clip(idx,0,bins-1), 1) #일반적인 표현: hist[idx] += 1

        # ----- 최종 계산 -----
        print("Calculate the results...")
        results = {}
        for sp, a in tqdm(acc.items(), desc='Processing2...'):
            if a["n"] == 0:
                continue

            mean_pred = a["sum_pred"]/a["n"]
            c = a["hist_pred"].cumsum()
            # histogram을 활용하여 분위수를 계산하는 함수
            def q(p):
                k = np.searchsorted(c, p*c[-1]) # p*c[-1]: p 위치의 순위
                return 0.5*(edges[k]+edges[min(k+1,bins)]) # k번째 값이 속하는 구간의 중위값
            results[sp] = dict(mean=mean_pred, median=q(0.5), Q2=q(0.25), Q3=q(0.75), minimum=acc[sp]["min"], maximum=acc[sp]["max"])
            
        return results, acc, edges

In [ ]:
age_path = r"F:/CBH/AGCLS_INT.tif"
dbh_path = r"F:/CBH/DMCLS_INT.tif"
category_compare, cat_acc, cat_edges = analyze_raster_categorywise(
    pred_path=pred_cbh, species_path=species_path, age_path=age_path, dbh_path=dbh_path,
    bins=20, nodata=None, vmin=0, vmax=30
)

In [ ]:
category_compare = {}
edges = cat_edges.copy()
bins=20
for sp, a in tqdm(cat_acc.items(), desc='Processing2...'):
    if a["n"] == 0:
        continue

    mean_pred = a["sum_pred"]/a["n"]
    c = a["hist_pred"].cumsum()
    # histogram을 활용하여 분위수를 계산하는 함수
    def q(p):
        k = np.searchsorted(c, p*c[-1]) # p*c[-1]: p 위치의 순위
        return 0.5*(edges[k]+edges[min(k+1,bins)]) # k번째 값이 속하는 구간의 중위값
    category_compare[sp] = dict(mean=mean_pred, median=q(0.5), Q2=q(0.25), Q3=q(0.75), minimum=cat_acc[sp]["min"], maximum=cat_acc[sp]["max"])

In [ ]:
df_cat = pd.DataFrame(category_compare).T.reset_index()
new_cols = ['SID', 'AGECLS', 'DMCLS']
new_data = df_cat['index'].apply(lambda x : x.split('-')).values
new_data = np.array([np.asarray(row) for row in new_data]).astype('int')
df_new = pd.concat([pd.DataFrame(columns=new_cols, data=new_data), df_cat], axis=1)
df_new.sort_values(by=['SID', 'AGECLS', 'DMCLS']).to_csv(os.path.join(save_dir, "Pred-NIFS-ComparebyCategory_plotdisjoint.csv"))

In [ ]:
category_compare.to_csv(os.path.join(save_dir, "Pred_byCategory_compare_plotdisjoint.csv"), index=False)

# plot별 대표값 산출(mean, median, std, IQR)

In [ ]:
# 포인트 시각화
def visualize_point(gdf):
    gdf = gdf.to_crs(epsg=3857) # basemap에 맞춰 좌표 변환
    fig, ax = plt.subplots(figsize=(10, 8))
    gdf.plot(ax=ax, color="blue", markersize=10, edgecolor="gray")
    cxt.add_basemap(ax=ax, source=cxt.providers.OpenStreetMap.Mapnik) # 배경지도 추가 # , zoom=10
    """
    ax.set_xlim(160000, 250000)
    ax.set_ylim(380000, 600000)
    """
    ax.set_title("Points (Republic of Korea)")
    plt.tight_layout()
    plt.show()
    
visualize_point(gdf_nfi7)

In [ ]:
gdf_nfi7 = gdf_nfi7.reset_index()
gdf_nfi7.head(3)

In [ ]:
# point에서 30m Buffer
import random

def generate_random_point(row, cnt):
    points = []
    minx, miny, maxx, maxy = row.bounds
    while len(points) < cnt:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if row.contains(p): points.append(p)
    return points

imsang_list = [19] # 19, 20, 21, 43, 45, 65, 66, 67, 68, 77
buffer_size = 30
n_points = 5
gdf_nfi7['buffer'] = gdf_nfi7.geometry.apply(lambda geom: geom.buffer(buffer_size)) # distance: raidus of the buffer, resolution: 원모양의 정밀도

random_points = []
parent_ids = []
for imsang_type in tqdm(imsang_list, desc="Extracting random points...", leave=True):
    select_condition = gdf_nfi7['SID'] == imsang_type
    for idx, row in tqdm(gdf_nfi7[select_condition].iterrows(), desc=f"Tree type: {imsang_type}", leave=False):
        points = generate_random_point(row['buffer'], 5)
        random_points.extend(points)
        parent_ids.extend([row['index']] * n_points)
    
gdf_random = gpd.GeoDataFrame({'parent_id' : parent_ids}, geometry=random_points, crs=gdf_nfi7.crs)

In [ ]:
gdf_proj = gdf_nfi7[gdf_nfi7['SID'] == 19].copy().to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 8))
gdf_proj.plot(ax=ax, color='blue', markersize=100, label='original')
gdf_proj.set_geometry('buffer').plot(ax=ax, facecolor='lightblue', edgecolor='gray', alpha=0.3)
gdf_random.plot(ax=ax, color='red', markersize=50, label='random sampled')

# Force appropriate limits for South Korea EPSG:5174
"""ax.set_xlim(150000, 300000)
ax.set_ylim(350000, 600000)
"""
ax.set_title("Randomly sampled points")
ax.legend()
plt.tight_layout()
plt.show()